In [21]:
#张量运算
import tensorflow as tf

print(tf.__version__)

#检查是否有GPU
print(tf.config.list_physical_devices('GPU'))

#声明常数
a = tf.constant([[1,2,3,4]])
print(a)
print(a + 10)
print(a - 10)
print(a / 10)
print(a * 10)

print(a.numpy())
print("===========================================\n")
#转为负数
print(tf.negative(a))
print("===========================================\n")

#做加法

#两个数
print(tf.add(1,2))
#两个向量相加
print(tf.add([1,2],[3,4]))

#平方
print(tf.square(5))
print(tf.square([1,2,3]))
print("=================计算和========================\n")
print(tf.reduce_sum([1,2,3]))
print(tf.reduce_sum(tf.constant([1,2,3])))
print(tf.reduce_sum([[1,2,5],[3,4,5]]))


2.18.0
[]
tf.Tensor([[1 2 3 4]], shape=(1, 4), dtype=int32)
tf.Tensor([[11 12 13 14]], shape=(1, 4), dtype=int32)
tf.Tensor([[-9 -8 -7 -6]], shape=(1, 4), dtype=int32)
tf.Tensor([[0.1 0.2 0.3 0.4]], shape=(1, 4), dtype=float64)
tf.Tensor([[10 20 30 40]], shape=(1, 4), dtype=int32)
[[1 2 3 4]]

tf.Tensor([[-1 -2 -3 -4]], shape=(1, 4), dtype=int32)

tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor([4 6], shape=(2,), dtype=int32)
tf.Tensor(25, shape=(), dtype=int32)
tf.Tensor([1 4 9], shape=(3,), dtype=int32)
=================计算和========================

tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(20, shape=(), dtype=int32)


In [4]:
#
import tensorflow as tf

'''
    tensorflow中的常数放在CPU上(使用constant方法),其他的放在GPU上
'''
x1 = tf.constant([1,2,3],dtype=tf.float32)
print("x1 是否在GPU #0 上:",x1.device.endswith('GPU:0')) # false,分配到cpu上

#设定x为均匀分配乱数 3x3
x2 = tf.random.uniform([3,3])
print("x2 是否在GPU #0 上:",x2.device.endswith('GPU:0')) # 如果有显卡,true

#
x3 = x1 + x2
print("x3 是否在GPU #0 上:",x3.device.endswith('GPU:0')) # 如果有显卡,true

x1 是否在GPU #0 上: False
x2 是否在GPU #0 上: False
x3 是否在GPU #0 上: False


In [34]:
#指定CPU或者GPU
import tensorflow as tf
import time

print(tf.config.list_physical_devices('CPU'))

#计算10次的时间
def time_matnul(x):
    start = time.time()
    for _ in range(10):
        tf.matmul(x,x)
    result = time.time() - start
    print("{:0.2f}ms".format(result*1000))

#强制在cpu上计算
print("在CPU上计算")
with tf.device("CPU:0"):
    x = tf.random.uniform([1000,1000])
    time_matnul(x)

#在GPU上计算
if tf.config.list_physical_devices('GPU'):
    print("在GPU上计算")
    with tf.device("GPU:0"):
        x = tf.random.uniform([1000,1000])
        time_matnul(x)


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
在CPU上计算
56.85ms


In [40]:
#稀疏矩阵(Sparser Matrix)
import tensorflow as tf
import numpy as np
'''
    1   0   0   0
    0   0   2   0
    0   0   0   0

'''
sparser_tensor = tf.sparse.SparseTensor(indices=[[0,0],[1,2]],values=[1,2],dense_shape=[3,4])
#打印值
print(sparser_tensor)

#转为正常的矩阵格式
x = tf.sparse.to_dense(sparser_tensor,default_value=0)
print(x)
print(type(x))
print(x.numpy())




SparseTensor(indices=tf.Tensor(
[[0 0]
 [1 2]], shape=(2, 2), dtype=int64), values=tf.Tensor([1 2], shape=(2,), dtype=int32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64))
tf.Tensor(
[[1 0 0 0]
 [0 0 2 0]
 [0 0 0 0]], shape=(3, 4), dtype=int32)
<class 'tensorflow.python.framework.ops.EagerTensor'>
[[1 0 0 0]
 [0 0 2 0]
 [0 0 0 0]]
2.0.2


In [2]:
import tensorflow as tf

if tf.__version__ != '1': #是否为tensorflow1.0版本
    import tensorflow.compat.v1 as tf # 改变导入框架的命名空间
    tf.disable_v2_behavior() # 禁用v2


Instructions for updating:
non-resource variables are not supported in the long term


In [4]:
import tensorflow as tf

#限制tensorflow只能使用2GB内存
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024 * 2)])

        #显示GPU个数
        logic_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(logic_gpus), "Physical GPUs",len(logic_gpus),"Logical GPUS")
    except RuntimeError as e:
        # 异常处理
        print(e)


In [5]:
import os
'''
设置为 -1 时，表示在程序运行过程中，CUDA（用于GPU加速计算的并行计算平台和编程模型）将看不到任何可用的GPU设备。
这可能是因为程序不需要GPU计算，或者在没有GPU的设备上运行该程序，设置这个变量可以避免程序尝试使用不存在的GPU资源而报错。
'''
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [17]:
'''
    自动微分
'''
import numpy as np
import tensorflow as tf

tf.compat.v1.enable_eager_execution()  # 在程序启动时启用 Eager Execution

print(tf.executing_eagerly())  # 应该输出 True

x = tf.Variable(3.0) #声明变量x

with tf.GradientTape() as g: #自动微分
    y = x * x

dy_dx = g.gradient(y,x) #计算梯度

print(dy_dx.numpy())

'''
    下面报错原因是tensorflow处于图执行模式
'''

ValueError: tf.enable_eager_execution must be called at program startup.

In [18]:
import numpy as np
import tensorflow as tf

x = tf.constant(3.0)
with tf.GradientTape() as g:
    g.watch(x) #设定参数参与自动微分
    y = x * x

dy_dx = g.gradient(y,x)

print(dy_dx.numpy())

AttributeError: 'SymbolicTensor' object has no attribute 'numpy'

In [ ]:
'''
    自动微分
'''
import numpy as np
import tensorflow as tf

# tf.compat.v1.enable_eager_execution()  # 在程序启动时启用 Eager Execution

print(tf.executing_eagerly())  # 应该输出 True

x = tf.Variable(3.0) #声明变量x

with tf.GradientTape() as g: #自动微分
    y = x * x

dy_dx = g.gradient(y,x) #计算梯度

print(dy_dx.numpy())

In [ ]:
#计算二阶导数

import tensorflow as tf

x = tf.Variable(3.0)

with tf.GradientTape() as g: #自动微分
    g.watch(x)
    with tf.GradientTape() as gg: #自动微分
        gg.watch(x) #设定参数参与自动微分
        y = x * x # y = x^ 2

    dy_dx = gg.gradient(y,x)
d2y_dx2 = g.gradient(dy_dx,x)

print(f'一阶导数:{dy_dx.numpy()} 二阶导数:{d2y_dx2.numpy()}')

In [ ]:
#多变量计算导数
import tensorflow as tf

x = tf.Variable(3.0)
with tf.GradientTape(persistent=True) as g: #自动微分
    y = x * x
    z = y * y

dz_dx = g.gradient(z,x) #4*x^3
dy_dx = g.gradient(y,x) #2*x

del g #不用时删除GradientTape对象

print(f'dy_dx:{dy_dx.numpy()} dz_dx:{dz_dx.numpy()}')